# CS5481 - Lecture 6 SQL Tutorial
## LLMs for Text-to-SQL and query processing

This short tutorial contains four guided examples and one practice question. It focuses only on SQL-related ideas shown in Lecture 6: Text-to-SQL, precise relational queries, query synthesis and execution, and query rewrite.


## 0. Preparation

This notebook follows the lecture's pipeline: **natural-language question -> SQL synthesis -> query execution -> result**. It uses DeepSeek through the OpenAI-compatible Chat Completions interface and SQLite for a small local database.


In [ ]:
!pip install -q openai pandas

import os
import re
import sqlite3
import pandas as pd
from openai import OpenAI


Store the API key outside the notebook. In Colab, create a secret named `DEEPSEEK_API_KEY`; locally, set the environment variable before starting Jupyter.


In [ ]:
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")

if DEEPSEEK_API_KEY is None:
    try:
        from google.colab import userdata
        DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
    except Exception:
        DEEPSEEK_API_KEY = ""

client = OpenAI(
    api_key=DEEPSEEK_API_KEY or "dummy",
    base_url="https://api.deepseek.com",
)
MODEL = "deepseek-chat"
print("DeepSeek key found:", DEEPSEEK_API_KEY != "")


### A small relational database

There are three tables: students, courses, and enrollments. The schema is included in every Text-to-SQL prompt because the model must know the available tables, columns, and relationships.


In [ ]:
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    major TEXT NOT NULL
);

CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    department TEXT NOT NULL
);

CREATE TABLE enrollments (
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    grade INTEGER NOT NULL,
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);

INSERT INTO students VALUES
    (1, 'Alice', 'Data Science'),
    (2, 'Bob', 'Computer Science'),
    (3, 'Carol', 'Data Science'),
    (4, 'David', 'Information Systems'),
    (5, 'Eva', 'Computer Science');

INSERT INTO courses VALUES
    (101, 'Database Systems', 'CS'),
    (102, 'Machine Learning', 'CS'),
    (103, 'Data Visualization', 'DS');

INSERT INTO enrollments VALUES
    (1, 101, 88), (1, 102, 91),
    (2, 101, 79), (2, 103, 84),
    (3, 101, 93), (3, 103, 90),
    (4, 101, 82),
    (5, 102, 86), (5, 103, 88);
""")

SCHEMA = """
students(student_id INTEGER PRIMARY KEY, name TEXT, major TEXT)
courses(course_id INTEGER PRIMARY KEY, course_name TEXT, department TEXT)
enrollments(student_id INTEGER, course_id INTEGER, grade INTEGER)
Relationships:
- enrollments.student_id = students.student_id
- enrollments.course_id = courses.course_id
""".strip()

pd.read_sql_query("SELECT * FROM students", conn)


### Reusable DeepSeek and SQL helpers

The prompt gives an explicit task, the schema, and a strict output format. `temperature=0` is used because SQL generation should be stable. Before execution, the notebook accepts only one read-only `SELECT` or `WITH` query.


In [ ]:
def ask(prompt, system=None, temperature=0.0, max_tokens=500):
    if DEEPSEEK_API_KEY == "":
        raise RuntimeError("Add DEEPSEEK_API_KEY before calling DeepSeek.")

    messages = []
    if system is not None:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()


def clean_sql(text):
    sql = re.sub(r"^```(?:sql)?\s*|\s*```$", "", text.strip(), flags=re.I)
    return sql.strip().rstrip(";")


def text_to_sql(question):
    prompt = f"""Convert the natural-language question into one SQLite query.

Database schema:
{SCHEMA}

Question: {question}

Rules:
- Use only the tables and columns in the schema.
- Return SQL only, with no Markdown and no explanation.
- Produce one read-only SELECT query.
"""
    return clean_sql(ask(prompt, system="You are a careful Text-to-SQL assistant."))


def run_query(sql):
    compact = " ".join(sql.split())
    if not re.match(r"^(SELECT|WITH)\b", compact, flags=re.I):
        raise ValueError("Only SELECT or WITH queries are allowed.")
    if ";" in compact:
        raise ValueError("Only one SQL statement is allowed.")
    print("Generated SQL:\n" + sql)
    return pd.read_sql_query(sql, conn)


## 1. Example 1: filtering with `SELECT` and `WHERE`

The model maps a natural-language condition to SQL's strict syntax. This is the simplest Text-to-SQL case from the lecture.


In [ ]:
question = "List the names of all students whose major is Data Science, ordered by name."
sql = text_to_sql(question)
run_query(sql)


## 2. Example 2: aggregation with `GROUP BY`

Natural language can describe an analytical request; SQL expresses it precisely using aggregation and grouping.


In [ ]:
question = "Count the number of students in each major. Show major and student_count, largest count first."
sql = text_to_sql(question)
run_query(sql)


## 3. Example 3: combining tables with `JOIN`

The schema relationships guide the model to connect students, enrollments, and courses.


In [ ]:
question = "Show every course taken by Alice and her grade. Return course_name and grade, highest grade first."
sql = text_to_sql(question)
run_query(sql)


## 4. Example 4: query rewrite

The lecture shows that prompting can rewrite a query while preserving its meaning. Here DeepSeek changes a subquery into an equivalent join-based query.


In [ ]:
original_sql = """
SELECT name
FROM students
WHERE student_id IN (
    SELECT student_id
    FROM enrollments
    WHERE grade >= 90
)
ORDER BY name
""".strip()

rewrite_prompt = f"""Rewrite the SQLite query below using a JOIN instead of IN.
Keep the result semantically equivalent and avoid duplicate student names.
Return SQL only, with no Markdown and no explanation.

Schema:
{SCHEMA}

Input SQL:
{original_sql}
"""

rewritten_sql = clean_sql(ask(
    rewrite_prompt,
    system="You rewrite SQL queries while preserving their meaning.",
))

print("Original result:")
display(run_query(original_sql))
print("Rewritten result:")
display(run_query(rewritten_sql))


## 5. Practice: Text-to-SQL with join, aggregation, and filtering

Ask DeepSeek:

> For each course, calculate the average grade. Return `course_name` and `average_grade`, keep only courses whose average grade is at least 85, and order from highest to lowest average grade.

Complete only the two marked lines. Then inspect the generated SQL and execute it. The expected output has two columns, but the SQL itself is produced by DeepSeek.


In [ ]:
exercise_question = (
    "For each course, calculate the average grade. Return course_name and average_grade, "
    "keep only courses whose average grade is at least 85, and order from highest to lowest average grade."
)

# TODO 1: Call the Text-to-SQL helper with exercise_question.
exercise_sql = ...

# TODO 2: Execute the generated SQL with the read-only query helper.
exercise_result = ...

exercise_result


## Summary

- Text-to-SQL converts a natural-language question into an executable relational query.
- The schema and explicit output rules make the prompt more reliable.
- SQL provides precise execution; always inspect generated SQL before using it on a real database.
- Query rewrite asks an LLM for an equivalent SQL expression, not a different answer.
